# Agentic Graph Retrieval with Two Tools

This notebook demonstrates a more advanced agentic setup. We will provide the Strands Agent with two distinct tools for interacting with the Graph RAG system:

1.  **`get_answer`**: Returns a final, synthesized answer.
2.  **`get_retrieved_contexts`**: Returns the raw, detailed source contexts with relevance scores.

This separation allows for more precise control and inspection of the retrieval process.

## 1. Setup

In [6]:
# Ensure the required libraries are installed
# !pip install -q strands-agents boto3

In [7]:
%reload_ext dotenv
%dotenv ../.env

import os
import json
from strands import Agent, tool
from strands.models import BedrockModel

from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory

# Set up logging for clarity
set_logging_config("WARNING")

## 2. Define Custom Tools for Graph Retrieval

We define two functions, each decorated with `@tool`.

### Tool 1: Get Final Answer

In [8]:
@tool
def get_answer(query: str) -> str:
    """
    Searches the knowledge graph and returns a final, synthesized answer to a question.
    Use this when you need a direct, summarized answer.

    Args:
        query (str): The question to ask the knowledge graph.

    Returns:
        str: The final answer from the knowledge graph, including source citations.
    """
    print(f"--- Calling Knowledge Graph for a final answer with query: {query} ---")
    
    graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
    vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store,
        vector_store,
        streaming=False
    )

    response = query_engine.query(query)
    
    return response.response

### Tool 2: Get Retrieved Contexts

In [9]:
@tool
def get_retrieved_contexts(query: str) -> str:
    """
    Searches the knowledge graph and returns the raw, detailed source contexts used to generate an answer.
    Use this when you need to see the exact pieces of information and their relevance scores for verification.

    Args:
        query (str): The question to ask the knowledge graph.

    Returns:
        str: A JSON string representing a list of retrieved contexts, including scores, topics, and statements.
    """
    print(f"--- Calling Knowledge Graph for contexts with query: {query} ---")

    graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
    vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store,
        vector_store,
        streaming=False
    )

    response = query_engine.query(query)

    contexts = []
    # The response object contains a list of source_nodes
    for node_with_score in response.source_nodes:
        node = node_with_score.node
        # We extract the relevant parts of the metadata
        context_data = {
            "overall_score": node_with_score.score if node_with_score.score is not None else 'N/A',
            "source_metadata": node.metadata.get('source', {}).get('metadata', {}),
            "topics": node.metadata.get('topics', [])
        }
        contexts.append(context_data)
    
    # Return the structured data as a JSON string
    return json.dumps(contexts, indent=2)

## 3. Create and Configure the Agent

In [10]:
# Define the system prompt to make the agent aware of its tools
system_prompt = """You are a helpful research assistant. You have two tools at your disposal:
1. `get_answer`: Use this to provide a direct, summarized answer to a user's question.
2. `get_retrieved_contexts`: Use this when the user explicitly asks for sources, details, context, or scores.

Analyze the user's request and choose the appropriate tool."""

# Configure the underlying Large Language Model (LLM)
model = BedrockModel(
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
)

# Create the agent with both tools
agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[
        get_answer,
        get_retrieved_contexts,
    ],
)

## 4. Invoke the Agent for a Final Answer

In [11]:
# Ask a question where a summarized answer is appropriate.
# The agent should choose the 'get_answer' tool.
question_for_answer = "What are statistical language models?"

response_answer = agent(question_for_answer)


Tool #1: get_answer
--- Calling Knowledge Graph for a final answer with query: What are statistical language models? ---


The key points are:

- Statistical language models view text as a sequence of words and estimate the probability of text as the product of the probabilities of individual words.
- The most common statistical language models are n-gram models, which estimate word probabilities based on counts of n-gram sequences in text corpora.
- N-gram models have limitations in fully capturing the complexity of natural language due to data sparsity and need to use smoothing techniques.
- Statistical language models are foundational to many NLP tasks like machine translation, information retrieval, and speech recognition.

## 5. Invoke the Agent for Detailed Contexts

In [12]:
# Ask a question where detailed context is requested.
# The agent should choose the 'get_retrieved_contexts' tool.
question_for_context = "Show me the detailed context and scores for the query: 'What are statistical language models?'"

response_context = agent(question_for_context)

# The agent's response will contain the JSON string from the tool. Let's parse and print it.
try:
    retrieved_data = json.loads(str(response_context))
    print(json.dumps(retrieved_data, indent=2))
except (json.JSONDecodeError, TypeError):
    print("Could not parse the response as JSON. Raw response:")
    print(response_context)


Here are the detailed retrieved contexts and scores for the query "What are statistical language models?":
Tool #2: get_retrieved_contexts
--- Calling Knowledge Graph for contexts with query: What are statistical language models? ---


The key points from the detailed retrieved contexts are:

- Statistical language models view text as a sequence of words and estimate the probability of text as the product of word probabilities. N-gram models are a widely used form of statistical language models.
- N-gram models have limitations in fully capturing the diversity of natural language due to data sparsity, and need to use smoothing techniques.
- Statistical language modeling is fundamental to many NLP tasks like machine translation, information retrieval, and speech recognition.
- Recent advances in transformer-based large language models (LLMs) have significantly extended the capabilities of language models.
- LLMs are much larger in model size compared to pre-trained language models (PLMs

## 6. Inspect the Agent's Full Conversation

In [13]:
# The agent's 'messages' attribute stores the entire conversation history.
# Note that the agent maintains a single history. The second call is a follow-up to the first.
print(json.dumps(agent.messages, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "What are statistical language models?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "toolUse": {
          "toolUseId": "tooluse_Ku_xokq1RiGO6aU_jN9X2A",
          "name": "get_answer",
          "input": {
            "query": "What are statistical language models?"
          }
        }
      }
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "toolResult": {
          "toolUseId": "tooluse_Ku_xokq1RiGO6aU_jN9X2A",
          "status": "success",
          "content": [
            {
              "text": "Statistical language models are a dominant form of language models that view text as a sequence of words and estimate the probability of text as the product of the probabilities of the individual words. [Source: data/pdfs/sample_Pdf.pdf (sample_Pdf.pdf, 1, 2, 1)]\n\nThe most widely used statistical language models are n-gram models, which estimate word probabiliti